In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import segmentation_models_pytorch as smp
import mediapipe as mp
from tqdm import tqdm


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


GAN_MODEL_PATH = r"D:\senior_project\result_v9_5channels\models\gen_epoch_15.pth" 
SEG_MODEL_PATH = r"D:\senior_project\best_unet_resnet34.pth" 


#INPUT_DIR = r"D:\senior_project\dataset\Meglass_evaluation\with_glasses"
INPUT_DIR = r"D:\senior_project\test_b4_pre"
#OUTPUT_DIR = r"D:\senior_project\evaluation_data\Meglass_results"
OUTPUT_DIR = r"D:\senior_project\result_b4_pre"

os.makedirs(OUTPUT_DIR, exist_ok=True)

class UNetDown(nn.Module):
    def __init__(self, in_size, out_size, normalize=True, dropout=0.0):
        super(UNetDown, self).__init__()
        layers = [nn.Conv2d(in_size, out_size, 4, 2, 1, bias=False)]
        if normalize: layers.append(nn.InstanceNorm2d(out_size))
        layers.append(nn.LeakyReLU(0.2))
        if dropout: layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x): return self.model(x)

class UNetUp(nn.Module):
    def __init__(self, in_size, out_size, dropout=0.0):
        super(UNetUp, self).__init__()
        layers = [nn.ConvTranspose2d(in_size, out_size, 4, 2, 1, bias=False), nn.InstanceNorm2d(out_size), nn.ReLU(inplace=True)]
        if dropout: layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x, skip):
        x = self.model(x); x = torch.cat((x, skip), 1); return x

class GeneratorUNet(nn.Module):
    def __init__(self, in_channels=5, out_channels=3): 
        super(GeneratorUNet, self).__init__()
        self.down1 = UNetDown(in_channels, 64, normalize=False)
        self.down2 = UNetDown(64, 128)
        self.down3 = UNetDown(128, 256)
        self.down4 = UNetDown(256, 512, dropout=0.5)
        self.down5 = UNetDown(512, 512, dropout=0.5)
        self.down6 = UNetDown(512, 512, dropout=0.5)
        self.down7 = UNetDown(512, 512, dropout=0.5)
        self.down8 = UNetDown(512, 512, normalize=False, dropout=0.5)
        self.up1 = UNetUp(512, 512, dropout=0.5); self.up2 = UNetUp(1024, 512, dropout=0.5)
        self.up3 = UNetUp(1024, 512, dropout=0.5); self.up4 = UNetUp(1024, 512, dropout=0.5)
        self.up5 = UNetUp(1024, 256); self.up6 = UNetUp(512, 128); self.up7 = UNetUp(256, 64)
        self.final = nn.Sequential(nn.Upsample(scale_factor=2), nn.ZeroPad2d((1, 0, 1, 0)), nn.Conv2d(128, out_channels, 4, padding=1), nn.Tanh())
    def forward(self, x):
        d1 = self.down1(x); d2 = self.down2(d1); d3 = self.down3(d2); d4 = self.down4(d3)
        d5 = self.down5(d4); d6 = self.down6(d5); d7 = self.down7(d6); d8 = self.down8(d7)
        u1 = self.up1(d8, d7); u2 = self.up2(u1, d6); u3 = self.up3(u2, d5); u4 = self.up4(u3, d4)
        u5 = self.up5(u4, d3); u6 = self.up6(u5, d2); u7 = self.up7(u6, d1); return self.final(u7)

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, refine_landmarks=True)
mp_face_detection = mp.solutions.face_detection
face_detection = mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5)

seg_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

gan_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def get_landmark_tensor_v9(image_pil):
    w, h = image_pil.size
    img_np = np.array(image_pil)
    landmark_map = np.zeros((h, w), dtype=np.uint8)
    results = face_mesh.process(img_np)
    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0].landmark
        def to_px(idx): return (int(landmarks[idx].x * w), int(landmarks[idx].y * h))
        
        LEFT = [362, 382, 381, 380, 374, 373, 390, 249, 263, 466, 388, 387, 386, 385, 384, 398]
        RIGHT = [33, 7, 163, 144, 145, 153, 154, 155, 133, 173, 157, 158, 159, 160, 161, 246]
        
        pts_left = np.array([to_px(i) for i in LEFT], np.int32).reshape((-1, 1, 2))
        cv2.polylines(landmark_map, [pts_left], True, 255, 2)
        pts_right = np.array([to_px(i) for i in RIGHT], np.int32).reshape((-1, 1, 2))
        cv2.polylines(landmark_map, [pts_right], True, 255, 2)
        
    landmark_pil = Image.fromarray(landmark_map).resize((256, 256))
    return transforms.ToTensor()(landmark_pil).unsqueeze(0).to(DEVICE)

# LOAD MODELS
print("Loading Models...")
seg_model = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
if os.path.exists(SEG_MODEL_PATH):
    seg_model.load_state_dict(torch.load(SEG_MODEL_PATH, map_location=DEVICE))
    seg_model.eval()

gen_model = GeneratorUNet(in_channels=5).to(DEVICE)
if os.path.exists(GAN_MODEL_PATH):
    gen_model.load_state_dict(torch.load(GAN_MODEL_PATH, map_location=DEVICE))
    gen_model.eval()


def process_face_roi(face_pil):
    """
    ฟังก์ชันนี้รับรูปหน้าขนาดจัตุรัส 256x256 เข้ามา แล้วส่งคืนรูปหน้าที่ลบแว่นแล้ว
    """
    # 1. Segmentation
    seg_input = seg_transform(face_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        mask_pred = (seg_model(seg_input) > 0).float()
    
    if mask_pred.sum() < 50: # ถ้า Mask น้อยมาก (หาแว่นไม่เจอ)
        return np.array(face_pil.resize((256, 256)))
        
    # 2. Mask Dilation
    mask_np = mask_pred.squeeze().cpu().numpy()
    kernel = np.ones((10, 10), np.uint8) 
    mask_dilated_np = cv2.dilate(mask_np, kernel, iterations=1)
    mask_tensor = torch.from_numpy(mask_dilated_np).unsqueeze(0).unsqueeze(0).to(DEVICE)
    
    # 3. GAN Inference (V9: 5 Channels)
    gan_img_tensor = gan_transform(face_pil).unsqueeze(0).to(DEVICE)
    
    masked_input = gan_img_tensor * (1 - mask_tensor)
    landmark_tensor = get_landmark_tensor_v9(face_pil)
    
    gan_input = torch.cat((masked_input, mask_tensor, landmark_tensor), 1)
    
    with torch.no_grad():
        fake_output = gen_model(gan_input)
        
    # 4. Post-Process (De-normalize)
    fake_output = (fake_output * 0.5) + 0.5
    gan_face_np = fake_output.squeeze().permute(1, 2, 0).cpu().numpy()
    gan_face_np = np.clip(gan_face_np * 255, 0, 255).astype(np.uint8)
    
    # 5. Seamless Clone (เพื่อให้ขอบเนียน)
    ori_face_np = np.array(face_pil.resize((256, 256)))
    mask_show = mask_tensor.squeeze().cpu().numpy()
    mask_np_uint8 = (mask_show * 255).astype(np.uint8)
    
    try:
        # ใช้ Seamless Clone เฉพาะบริเวณหน้ากาก
        coords = np.argwhere(mask_np_uint8 > 0)
        if len(coords) > 0:
            y_min, x_min = coords.min(axis=0); y_max, x_max = coords.max(axis=0)
            center = ((x_min + x_max) // 2, (y_min + y_max) // 2)
            final_blended = cv2.seamlessClone(gan_face_np, ori_face_np, mask_np_uint8, center, cv2.NORMAL_CLONE)
        else:
            final_blended = gan_face_np
    except:
        final_blended = gan_face_np # Fallback ถ้า Clone ไม่ได้
    
    return final_blended

def process_full_image(img_path):
    """
    ฟังก์ชันหลัก: อ่านรูปเต็ม -> Smart Square Crop -> GAN -> Paste Back
    """
    if not os.path.exists(img_path): return None
    
    # อ่านรูปด้วย OpenCV (จะได้จัดการเรื่อง Pad ง่ายๆ)
    full_img_cv2 = cv2.imread(img_path)
    if full_img_cv2 is None: return None
    full_img_rgb = cv2.cvtColor(full_img_cv2, cv2.COLOR_BGR2RGB)
    h_full, w_full = full_img_rgb.shape[:2]

    # 1. Face Detection
    results = face_detection.process(full_img_rgb)
    
    # ถ้าหาหน้าไม่เจอเลย คืนรูปเดิม
    if not results.detections:
        print(f"No face detected in {os.path.basename(img_path)}")
        return full_img_rgb

    # เอาหน้าที่มั่นใจที่สุด
    detection = results.detections[0]
    bbox = detection.location_data.relative_bounding_box
    
    # แปลงเป็น Pixel Coordinate
    x = int(bbox.xmin * w_full)
    y = int(bbox.ymin * h_full)
    w = int(bbox.width * w_full)
    h = int(bbox.height * h_full)

    # 2. Smart Square Calculation
    center_x, center_y = x + w // 2, y + h // 2
    
    # หาด้านที่ยาวที่สุด แล้ว Padding เพื่อให้เห็นหัวและคางครบ
    # ขยาย 50% จากกรอบหน้าเดิม
    max_dim = int(max(w, h) * 1.5) 
    
    # คำนวณกรอบสี่เหลี่ยมจัตุรัส
    crop_x1 = center_x - max_dim // 2
    crop_y1 = center_y - max_dim // 2
    crop_x2 = crop_x1 + max_dim
    crop_y2 = crop_y1 + max_dim
    
    # 3. Handle Out of Bounds (Smart Padding)
    # ถ้ากรอบหลุดออกนอกรูป เราจะ Pad รูปใหญ่ ด้วยสีดำ
    pad_left = max(0, -crop_x1)
    pad_top = max(0, -crop_y1)
    pad_right = max(0, crop_x2 - w_full)
    pad_bottom = max(0, crop_y2 - h_full)
    
    # สร้าง Canvas ใหม่ที่ใหญ่ขึ้น (ถ้าจำเป็น)
    if pad_left > 0 or pad_top > 0 or pad_right > 0 or pad_bottom > 0:
        full_img_padded = cv2.copyMakeBorder(full_img_rgb, pad_top, pad_bottom, pad_left, pad_right, cv2.BORDER_CONSTANT, value=[0,0,0])
        # ขยับพิกัด Crop ให้ตรงกับรูปใหม่ที่ Pad แล้ว
        crop_x1 += pad_left
        crop_y1 += pad_top
        crop_x2 += pad_left
        crop_y2 += pad_top
    else:
        full_img_padded = full_img_rgb

    # 4. Crop & Process
    face_crop = full_img_padded[crop_y1:crop_y2, crop_x1:crop_x2]
    
    # แปลงเป็น PIL เพื่อส่งเข้าฟังก์ชัน GAN เดิม
    face_pil = Image.fromarray(face_crop).resize((256, 256))
    
    # === ส่งเข้า GAN ===
    res_face_np = process_face_roi(face_pil) 
    # =================
    
    # 5. Paste Back (แปะกลับ)
    # Resize ผลลัพธ์กลับไปเท่าขนาดที่ Crop มา (max_dim x max_dim)
    res_face_resized = cv2.resize(res_face_np, (crop_x2 - crop_x1, crop_y2 - crop_y1))
    
    # แปะกลับลงไปในรูปที่ Pad แล้ว
    full_img_padded[crop_y1:crop_y2, crop_x1:crop_x2] = res_face_resized
    
    # ตัดส่วน Padding ออก เพื่อให้ได้ขนาดเท่ารูปต้นฉบับ
    final_h, final_w = full_img_padded.shape[:2]
    final_output = full_img_padded[pad_top:final_h-pad_bottom, pad_left:final_w-pad_right]
    
    # กรณีไม่มี Pad เลย
    if pad_left == 0 and pad_top == 0 and pad_right == 0 and pad_bottom == 0:
        final_output = full_img_padded
        
    return final_output

# EXECUTION
image_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.png', '.jpg')) and "_mask" not in f]
print(f"Processing V9 (End-to-End Smart Crop): {len(image_files)} images...")

success_count = 0
for filename in tqdm(image_files):
    try:
        img_path = os.path.join(INPUT_DIR, filename)
        result_img = process_full_image(img_path)
        
        if result_img is not None:
            save_path = os.path.join(OUTPUT_DIR, filename)
            Image.fromarray(result_img).save(save_path)
            success_count += 1
            
    except Exception as e:
        print(f"Error {filename}: {e}")

print(f"All Done! Processed {success_count}/{len(image_files)} images.")
print(f"Results saved to: {OUTPUT_DIR}")

Using device: cuda
Loading Models...


C:\Users\Asus\AppData\Local\Temp\ipykernel_21132\46754353.py:111: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  seg_model.load_state_dict(torch.load(SEG_MODEL_PATH, map_loca

Processing V9 (End-to-End Smart Crop): 3 images...


  0%|          | 0/3 [00:00<?, ?it/s]c:\Users\Asus\anaconda3\envs\gan_project\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
100%|██████████| 3/3 [00:01<00:00,  1.70it/s]

All Done! Processed 3/3 images.
Results saved to: D:\senior_project\result_b4_pre


In [6]:
!python -m pytorch_fid "D:\senior_project\evaluation_data\ground_truth_synthetic_resized" "D:\senior_project\evaluation_data\generatedsyn_v9_myproject_resized" --device cuda:0 --num-workers 0

FID:  16.103040661566638



100%|██████████| 6/6 [00:04<00:00,  1.27it/s]

100%|██████████| 6/6 [00:03<00:00,  1.52it/s]
